In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Grid size
n = 100

# Base forest
base_forest = np.ones((n, n), dtype=int)
base_forest[0, :] = base_forest[-1, :] = 0
base_forest[:, 0] = base_forest[:, -1] = 0

def classify_forest(forest):
    nf = (forest == 0)
    f  = (forest == 1)
    edge     = np.zeros_like(forest, dtype=bool)
    interior = np.zeros_like(forest, dtype=bool)

    for i in range(1, forest.shape[0]-1):
        for j in range(1, forest.shape[1]-1):
            if f[i, j]:
                if f[i-1,j] and f[i+1,j] and f[i,j-1] and f[i,j+1]:
                    interior[i,j] = True
                else:
                    edge[i,j] = True

    out = np.zeros_like(forest, dtype=int)
    out[nf]       = 0
    out[edge]     = 1
    out[interior] = 2
    return out, edge, interior

def scenario_block_clearcut():
    """
    情景 A：一大块贴着边界的 clearcut（低 EFCR，大面积损失）
    """
    prev_forest = base_forest.copy()
    prev_class, prev_edge, prev_interior = classify_forest(prev_forest)

    forest = prev_forest.copy()

    # 这里做一块大矩形，贴着左边界，你可以调节这几个参数
    top    = 10
    bottom = 80    # 不含 bottom 行
    left   = 1
    right  = 50    # 不含 right 列

    forest[top:bottom, left:right] = 0

    class_grid, edge_mask, interior_mask = classify_forest(forest)

    loss_mask = (prev_forest == 1) & (forest == 0)
    loss_area = loss_mask.sum()

    int2edge = (prev_interior & edge_mask).sum()
    efcr = int2edge / loss_area if loss_area > 0 else 0.0

    return {
        "prev_class": prev_class,
        "class_grid": class_grid,
        "loss_area": int(loss_area),
        "int2edge": int(int2edge),
        "efcr": float(efcr)
    }

def scenario_patchy_logging(num_patches=8, patch_h=8, patch_w=8, seed=0):
    """
    情景 B：多块零散小斑块伐木（高 EFCR，损失面积可以比 A 小/相近）
    """
    rng = np.random.default_rng(seed)

    prev_forest = base_forest.copy()
    prev_class, prev_edge, prev_interior = classify_forest(prev_forest)

    forest = prev_forest.copy()

    # 在内部随机放若干小块（避免太靠边界）
    margin = 10
    for _ in range(num_patches):
        i0 = rng.integers(margin, n - margin - patch_h)
        j0 = rng.integers(margin, n - margin - patch_w)
        forest[i0:i0+patch_h, j0:j0+patch_w] = 0

    class_grid, edge_mask, interior_mask = classify_forest(forest)

    loss_mask = (prev_forest == 1) & (forest == 0)
    loss_area = loss_mask.sum()

    int2edge = (prev_interior & edge_mask).sum()
    efcr = int2edge / loss_area if loss_area > 0 else 0.0

    return {
        "prev_class": prev_class,
        "class_grid": class_grid,
        "loss_area": int(loss_area),
        "int2edge": int(int2edge),
        "efcr": float(efcr)
    }

def search_patchy_logging(target_int2edge, tol=20,
                          num_patches_list=(6, 8, 10, 12),
                          patch_size_list=(6, 8, 10),
                          max_seed=200):
    """
    搜索一个 Scenario B，使得 Interior→Exterior 接近 target_int2edge，
    但 Loss area 明显小于 block clearcut。
    """
    best = None

    for num_patches in num_patches_list:
        for patch_h in patch_size_list:
            patch_w = patch_h
            for seed in range(max_seed):
                scenario = scenario_patchy_logging(
                    num_patches=num_patches,
                    patch_h=patch_h,
                    patch_w=patch_w,
                    seed=seed
                )
                int2edge = scenario["int2edge"]
                loss_area = scenario["loss_area"]

                # 要求 Interior→Exterior 靠近 target_int2edge，且损失面积更小
                if abs(int2edge - target_int2edge) == 0 and loss_area < scenarioA["loss_area"]:
                    best = (scenario, num_patches, patch_h, seed)
                    print("Found candidate:")
                    print("  num_patches =", num_patches,
                          "patch_h =", patch_h,
                          "seed =", seed)
                    print("  Loss =", scenario["loss_area"],
                          "Int→Ext =", scenario["int2edge"],
                          "EFCR =", scenario["efcr"])
                    return best  # 找到一个就直接返回

    return best

# target = Scenario A 的 Interior→Exterior
target_int2edge = scenarioA["int2edge"]
result = search_patchy_logging(target_int2edge)

if result is not None:
    scenarioB_tuned, num_patches, patch_h, seed = result
    print("\nUsing tuned Scenario B:")
    print("  num_patches =", num_patches, "patch_h =", patch_h, "seed =", seed)
    print("  Loss =", scenarioB_tuned["loss_area"])
    print("  Int→Ext =", scenarioB_tuned["int2edge"])
    print("  EFCR =", scenarioB_tuned["efcr"])
else:
    print("No suitable scenario found, try relaxing tol or expanding search ranges.")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import matplotlib.colors as mcolors

# Define custom colors for classes (0,1,2)
cmap = {
    0: "#D9D9D9",  # non-forest
    1: "#2E7D32",  # edge forest
    2: "#A5D6A7",  # interior forest
}

def apply_cmap(class_grid):
    """Convert class_grid (0/1/2) to an RGB image."""
    h, w = class_grid.shape
    rgb = np.zeros((h, w, 3))
    for cls, color in cmap.items():
        rgb[class_grid == cls] = mcolors.to_rgb(color)
    return rgb

# Convert grids to RGB
init_rgb  = apply_cmap(scenarioA["prev_class"])
A_rgb     = apply_cmap(scenarioA["class_grid"])
B_rgb     = apply_cmap(scenarioB_tuned["class_grid"])

fig, axes = plt.subplots(1, 3, figsize=(11, 4), dpi=600)

# --------------------------------------------
# Panel 1 — Initial forest
# --------------------------------------------
axes[0].imshow(init_rgb)
axes[0].set_title("Initial Forest", fontsize=12, fontweight='bold')
axes[0].axis("off")

# --------------------------------------------
# Panel 2 — Scenario A
# --------------------------------------------
axes[1].imshow(A_rgb)
axes[1].set_title("Scenario A:\nBlock Clearcut (Low EFCR)", fontsize=12, fontweight='bold')
axes[1].axis("off")
axes[1].text(
    0.67, 0.03,
    f"Loss = {scenarioA['loss_area']}\n"
    f"Int→Ext = {scenarioA['int2edge']}\n"
    f"EFCR ≈ {scenarioA['efcr']:.2f}",
    transform=axes[1].transAxes,
    fontsize=9,
    fontweight='bold',
    verticalalignment="bottom",
    bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=2)
)

# --------------------------------------------
# Panel 3 — Scenario B
# --------------------------------------------
axes[2].imshow(B_rgb)
axes[2].set_title("Scenario B:\nPatchy Logging (High EFCR)", fontsize=12, fontweight='bold')
axes[2].axis("off")
axes[2].text(
    0.67, 0.03,
    f"Loss = {scenarioB_tuned['loss_area']}\n"
    f"Int→Ext = {scenarioB_tuned['int2edge']}\n"
    f"EFCR ≈ {scenarioB_tuned['efcr']:.2f}",
    transform=axes[2].transAxes,
    fontsize=9,
    fontweight='bold',
    verticalalignment="bottom",
    bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=2)
)

# --------------------------------------------
# Add a global legend (bottom center)
# --------------------------------------------
legend_elements = [
    mpatches.Patch(facecolor=cmap[2], label='Interior forest'),
    mpatches.Patch(facecolor=cmap[1], label='Edge forest'),
    mpatches.Patch(facecolor=cmap[0], label='Non-forest'),
]

fig.legend(
    handles=legend_elements, 
    loc='lower center',
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, -0.05),
    fontsize=10
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Grid size
n = 100

# Base forest
base_forest = np.ones((n, n), dtype=int)
base_forest[0, :] = base_forest[-1, :] = 0
base_forest[:, 0] = base_forest[:, -1] = 0

def classify_forest(forest):
    nf = (forest == 0)
    f  = (forest == 1)
    edge     = np.zeros_like(forest, dtype=bool)
    interior = np.zeros_like(forest, dtype=bool)

    for i in range(1, forest.shape[0]-1):
        for j in range(1, forest.shape[1]-1):
            if f[i, j]:
                if f[i-1,j] and f[i+1,j] and f[i,j-1] and f[i,j+1]:
                    interior[i,j] = True
                else:
                    edge[i,j] = True

    out = np.zeros_like(forest, dtype=int)
    out[nf]       = 0
    out[edge]     = 1
    out[interior] = 2
    return out, edge, interior

def scenario_EFCR_1():
    """
    Scenario 1: 大块 clearcut 贴着边界（低 EFCR）
    20x25 = 500 像素
    """
    prev_forest = base_forest.copy()
    prev_class, prev_edge, prev_interior = classify_forest(prev_forest)

    forest = prev_forest.copy()
    top, bottom = 10, 30       # 高度 20
    left, right = 1, 26        # 宽度 25

    forest[top:bottom, left:right] = 0  # 20*25 = 500

    class_grid, edge_mask, interior_mask = classify_forest(forest)
    loss_mask = (prev_forest == 1) & (forest == 0)
    loss_area = loss_mask.sum()
    int2edge  = (prev_interior & edge_mask).sum()
    efcr      = int2edge / loss_area if loss_area > 0 else 0.0

    return {
        "prev_class": prev_class,
        "class_grid": class_grid,
        "loss_area": int(loss_area),
        "int2edge": int(int2edge),
        "efcr": float(efcr),
        "label": "Scenario 1"
    }

def scenario_EFCR_2():
    """
    Scenario 2: 同样大小的矩形，但整体往内移一段距离（中等偏低 EFCR）
    仍然是 20x25 = 500 像素
    """
    prev_forest = base_forest.copy()
    prev_class, prev_edge, prev_interior = classify_forest(prev_forest)

    forest = prev_forest.copy()
    top, bottom = 10, 30       # 高度 20
    left, right = 10, 35       # 宽度 25，离边界更远

    forest[top:bottom, left:right] = 0

    class_grid, edge_mask, interior_mask = classify_forest(forest)
    loss_mask = (prev_forest == 1) & (forest == 0)
    loss_area = loss_mask.sum()
    int2edge  = (prev_interior & edge_mask).sum()
    efcr      = int2edge / loss_area if loss_area > 0 else 0.0

    return {
        "prev_class": prev_class,
        "class_grid": class_grid,
        "loss_area": int(loss_area),
        "int2edge": int(int2edge),
        "efcr": float(efcr),
        "label": "Scenario 2"
    }

def scenario_EFCR_3():
    """
    Scenario 3: 两块内部矩形（更高 EFCR）
    两块 10x25 矩形，共 2*10*25=500
    """
    prev_forest = base_forest.copy()
    prev_class, prev_edge, prev_interior = classify_forest(prev_forest)

    forest = prev_forest.copy()

    # 左边内部块
    top1, bottom1 = 20, 30    # 10 高
    left1, right1 = 10, 35    # 25 宽

    # 右边内部块
    top2, bottom2 = 50, 60    # 10 高
    left2, right2 = 40, 65    # 25 宽

    forest[top1:bottom1, left1:right1] = 0
    forest[top2:bottom2, left2:right2] = 0   # 共 500 像素

    class_grid, edge_mask, interior_mask = classify_forest(forest)
    loss_mask = (prev_forest == 1) & (forest == 0)
    loss_area = loss_mask.sum()
    int2edge  = (prev_interior & edge_mask).sum()
    efcr      = int2edge / loss_area if loss_area > 0 else 0.0

    return {
        "prev_class": prev_class,
        "class_grid": class_grid,
        "loss_area": int(loss_area),
        "int2edge": int(int2edge),
        "efcr": float(efcr),
        "label": "Scenario 3"
    }

def scenario_EFCR_4():
    """
    Scenario 4: 多个小斑块散布内部（最高 EFCR）
    例如 5x5 的小块，共 500/25 = 20 个 patch
    """
    rng = np.random.default_rng(42)

    prev_forest = base_forest.copy()
    prev_class, prev_edge, prev_interior = classify_forest(prev_forest)

    forest = prev_forest.copy()
    patch_h = patch_w = 5
    num_patches = 20  # 20*25 = 500 像素

    margin = 10
    for _ in range(num_patches):
        i0 = rng.integers(margin, n - margin - patch_h)
        j0 = rng.integers(margin, n - margin - patch_w)
        forest[i0:i0+patch_h, j0:j0+patch_w] = 0

    class_grid, edge_mask, interior_mask = classify_forest(forest)
    loss_mask = (prev_forest == 1) & (forest == 0)
    loss_area = loss_mask.sum()
    int2edge  = (prev_interior & edge_mask).sum()
    efcr      = int2edge / loss_area if loss_area > 0 else 0.0

    return {
        "prev_class": prev_class,
        "class_grid": class_grid,
        "loss_area": int(loss_area),
        "int2edge": int(int2edge),
        "efcr": float(efcr),
        "label": "Scenario 4"
    }

sc1 = scenario_EFCR_1()
sc2 = scenario_EFCR_2()
sc3 = scenario_EFCR_3()
sc4 = scenario_EFCR_4()

for i, sc in enumerate([sc1, sc2, sc3, sc4], start=1):
    print(f"Scenario {i}:")
    print("  Loss area        =", sc["loss_area"])
    print("  Interior→Exterior=", sc["int2edge"])
    print("  EFCR             =", sc["efcr"])
    print()


In [ ]:
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

# 如果还没定义 cmap 和 apply_cmap：
cmap = {
    0: "#D9D9D9",  # non-forest
    1: "#2E7D32",  # edge forest
    2: "#A5D6A7",  # interior forest
}

def apply_cmap(class_grid):
    h, w = class_grid.shape
    rgb = np.zeros((h, w, 3))
    for cls, color in cmap.items():
        rgb[class_grid == cls] = mcolors.to_rgb(color)
    return rgb

# 初始森林
init_class, _, _ = classify_forest(base_forest)
init_rgb = apply_cmap(init_class)

scenarios = [sc1, sc2, sc3, sc4]

fig, axes = plt.subplots(1, 5, figsize=(16, 4), dpi=400)

# Panel 0 — Initial
axes[0].imshow(init_rgb)
axes[0].set_title("Initial\nForest", fontsize=11, fontweight='bold')
axes[0].axis("off")

# Panels 1–4 — EFCR increasing
for i, sc in enumerate(scenarios, start=1):
    ax = axes[i]
    rgb = apply_cmap(sc["class_grid"])
    ax.imshow(rgb)
    ax.axis("off")
    ax.set_title(
        f"{sc['label']}\nEFCR ≈ {sc['efcr']:.2f}",
        fontsize=11,
        fontweight='bold'
    )
    ax.text(
        0.67, 0.03,
        f"Loss = {sc['loss_area']}\n"
        f"Int→Ext = {sc['int2edge']}",
        transform=ax.transAxes,
        fontsize=8,
        fontweight='bold',
        verticalalignment="bottom",
        bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=2)
    )

# Legend
legend_elements = [
    mpatches.Patch(facecolor=cmap[2], label='Interior forest'),
    mpatches.Patch(facecolor=cmap[1], label='Edge forest'),
    mpatches.Patch(facecolor=cmap[0], label='Non-forest'),
]

fig.legend(
    handles=legend_elements,
    loc='lower center',
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, -0.08),
    fontsize=10
)

plt.tight_layout()
plt.show()